# FiveAM

# Example: FiveAM Example (poor man's tutorial) 

In [4]:
(ql:quickload "fiveam")

(in-package :cl-user)
(defpackage :it.bese.fiveam.example
  (:use :common-lisp
        :it.bese.fiveam))

(in-package :it.bese.fiveam.example)

;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
; function to test
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
(defun add-2 (n)
  (+ n 2))

(defun add-4 (n)
  (+ n 4))

(defun dummy-add (a b)
  (+ a b))
(defun dummy-strcat (a b)
  (concatenate 'string a b))

;
; root suite
;
(def-suite example-suite-root :description "The root example test suite.")


(def-suite* example-suite0 :in example-suite-root)

;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
; tests
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
(test add-2
  "Test the ADD-2 function" ;; a short description
  ;; the checks
  (is (= 2 (add-2 0)))
  (is (= 0 (add-2 -2))))

; it.bese.fiveam.example>(run! 'add-2)
; Running test add-2 ..
;   Did 2 checks.
;      Pass: 2 (100%)
;      Skip: 0 ( 0%)
;      Fail: 0 ( 0%)
; t
; nil
; nil

;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
; suites
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
(def-suite example-suite
           :description "The example test suite."
           :in example-suite-root)
(in-suite example-suite)
; or
; (def-suite* example-suite :in example-suite-root)

(test add-4
  (is (= 0 (add-4 -4))))

; it.bese.fiveam.example>(run! 'example-suite)
; Running test suite example-suite
;   Didn't run anything...huh?
; t
; nil
; nil

(test add-2 "Test the ADD-2 function"
  (is (= 2 (add-2 0)))
  (is (= 0 (add-2 -2))))
; fail case
; (test add-2 "Test the ADD-2 function"
;   (is (= 2 (add-2 0)))
;   (is (= 0 (add-2 -2)))
;   (is (= 0 (add-2 0))))

; it.bese.fiveam.example>(run! 'add-2)
; Running test add-2 ..f
;   Did 3 checks.
;      Pass: 2 (66%)
;      Skip: 0 ( 0%)
;      Fail: 1 (33%)
;   Failure Details:
;   --------------------------------
;   add-2 in example-suite [Test the ADD-2 function]: 

; (add-2 0)
;  evaluated to 
; 2
;  which is not 
; =
;  to 
; 0
;   --------------------------------
; nil
; (#<test-failure {11049314C3}>)
; nil


;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
; specification based testing
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;

(test dummy-add
  (for-all ((a (gen-integer))
            (b (gen-integer)))
    ;; assuming we have an "oracle" to compare our function results to
    ;; we can use it:
    (is (= (+ a b) (dummy-add a b)))
    ;; if we don't have an oracle (as in most cases) we just ensure
    ;; that certain properties hold:
    (is (= (dummy-add a b)
          (dummy-add b a)))
    (is (= a (dummy-add a 0)))
    (is (= 0 (dummy-add a (- a))))
    (is (< a (dummy-add a 1)))
    (is (= (* 2 a) (dummy-add a a)))))

(test dummy-strcat
  (for-all ((result (gen-string))
            (split-point (gen-integer :min 0 :max 10000)
                         (< split-point (length result))))
    (is (string= result (dummy-strcat (subseq result 0 split-point)
                                      (subseq result split-point))))))

; may fail
; (test random-failure
;   (for-all ((result (gen-integer :min 0 :max 1)))
;     (is (plusp result))
;     (is (= result 0))))



(run! 'example-suite-root)

To load "fiveam":
  Load 1 ASDF system:
    fiveam


("fiveam")

#<PACKAGE "COMMON-LISP-USER">

#<PACKAGE "IT.BESE.FIVEAM.EXAMPLE">

#<PACKAGE "IT.BESE.FIVEAM.EXAMPLE">

...

DUMMY-STRCAT

T

NIL

NIL

; Loading "fiveam"


Running test suite EXAMPLE-SUITE-ROOT
 Running test suite EXAMPLE-SUITE0
  Running test ADD-2 ..
 Running test suite EXAMPLE-SUITE
  Running test ADD-4 .
  Running test DUMMY-ADD .........................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................
  Running test DUMMY-STRCAT ....................................
 Did 5 checks.
    Pass: 5 (100%)
    Skip: 0 ( 0%)
    Fail: 0 ( 0%)



# Example: Common Lisp Cookbook - 35. Testing the code

In [2]:
(ql:quickload "fiveam")

;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
;;; Functions to Be Tested
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
;; We have a custom "file doesn't exist" condition.
(define-condition file-not-existing-error (error)
  ((filename :type string :initarg :filename :reader filename)))

;; We have a function that tries to read a file and signals the above condition
;; if the file doesn't exist.
(defun read-file-as-string (filename &key (error-if-not-exists t))
  "Read file content as string. FILENAME specifies the path of file.

Keyword ERROR-IF-NOT-EXISTS specifies the operation to perform when the file
is not found. T (by default) means an error will be signaled. When given NIL,
the function will return NIL in that case."
  (cond
    ((uiop:file-exists-p filename)
     (uiop:read-file-string filename))
    (error-if-not-exists
     (error 'file-not-existing-error :filename filename))
    (t nil)))

;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
;;; Define Suites
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
(in-package :cl-user)
(defpackage my-fiveam-test
  (:use :cl
        :fiveam))
(in-package :my-fiveam-test)

(def-suite my-system
  :description "Test my system")

(def-suite read-file-as-string
  :description "Test the read-file-as-string function."
  :in my-system)
(in-suite read-file-as-string)

;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
;;; Define Tests
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
;; Our first "base" case: we read a file that contains "hello".
(test read-file-as-string-normal-file
  (let ((result (read-file-as-string "hello.txt")))
    ;; Tip: put the expected value as the first argument of = or equal, string= etc.
    ;; FiveAM generates a more readable report following this convention.
    (is (string= "Hello, FiveAM!" result))))

;; We read an empty file.
(test read-file-as-string-empty-file
  (let ((result (read-file-as-string "empty.txt")))
    (is (not (null result)))
    ;; The reason can be used to provide formatted text.
    (is (= 0 (length result)))
        "Empty string expected but got ~a" result))

;; Now we test that reading a non-existing file signals our condition.
(test read-file-as-string-non-existing-file
  (let ((result (read-file-as-string "non-existing-file.txt"
                                     :error-if-not-exists nil)))
    (is (null result)
      "Reading a file should return NIL when :ERROR-IF-NOT-EXISTS is set to NIL"))
  ;; SIGNALS accepts the unquoted name of a condition and a body to evaluate.
  ;; Here it checks if FILE-NOT-EXISTING-ERROR is signaled.
  (signals file-not-existing-error
    (read-file-as-string "non-existing-file.txt"
                         :error-if-not-exists t)))

;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
; Custom Reasons
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
(fiveam:test simple-maths
  (is (= 3 (+ 1 1))
      "Maths should work, right? ~a. Another parameter is: ~S" t :foo))

;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
;;; Run Tests
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;
(run! 'my-system)


To load "fiveam":
  Load 1 ASDF system:
    fiveam


("fiveam")

FILE-NOT-EXISTING-ERROR

READ-FILE-AS-STRING

#<PACKAGE "COMMON-LISP-USER">

...

; in: ALEXANDRIA:NAMED-LAMBDA %TEST-READ-FILE-AS-STRING-NORMAL-FILE
;     (MY-FIVEAM-TEST::READ-FILE-AS-STRING "hello.txt")
; 
; caught STYLE-WARNING:
;   undefined function: MY-FIVEAM-TEST::READ-FILE-AS-STRING
; 
; compilation unit finished
;   Undefined function:
;     READ-FILE-AS-STRING
;   caught 1 STYLE-WARNING condition
; in: ALEXANDRIA:NAMED-LAMBDA %TEST-READ-FILE-AS-STRING-EMPTY-FILE
;     (MY-FIVEAM-TEST::READ-FILE-AS-STRING "empty.txt")
; 
; caught STYLE-WARNING:
;   undefined function: MY-FIVEAM-TEST::READ-FILE-AS-STRING
; 
; compilation unit finished
;   Undefined function:
;     READ-FILE-AS-STRING
;   caught 1 STYLE-WARNING condition
; in: ALEXANDRIA:NAMED-LAMBDA %TEST-READ-FILE-AS-STRING-NON-EXISTING-FILE
;     (IT.BESE.FIVEAM:SIGNALS MY-FIVEAM-TEST::FILE-NOT-EXISTING-ERROR
;       (MY-FIVEAM-TEST::READ-FILE-AS-STRING "non-existing-file.txt"
;        :ERROR-IF-NOT-EXISTS T))
; --> HANDLER-BIND SB-KERNEL::%HANDLER-BIND LET CONS 
; ==>
;   1
; 
; caught STYLE-WARNING:
;  

NIL

(#<IT.BESE.FIVEAM::UNEXPECTED-TEST-FAILURE {110574FE73}>
 #<IT.BESE.FIVEAM::UNEXPECTED-TEST-FAILURE {11057B0983}>
 #<IT.BESE.FIVEAM::UNEXPECTED-TEST-FAILURE {11057B0EF3}>
 #<IT.BESE.FIVEAM::TEST-FAILURE {11057B2983}>)

NIL


Running test suite MY-SYSTEM
 Running test suite READ-FILE-AS-STRING
  Running test READ-FILE-AS-STRING-NORMAL-FILE X
  Running test READ-FILE-AS-STRING-EMPTY-FILE X
  Running test READ-FILE-AS-STRING-NON-EXISTING-FILE X
  Running test SIMPLE-MATHS f
 Did 4 checks.
    Pass: 0 ( 0%)
    Skip: 0 ( 0%)
    Fail: 4 (100%)

 Failure Details:
 --------------------------------
 SIMPLE-MATHS in READ-FILE-AS-STRING []: 
      Maths should work, right? T. Another parameter is: :FOO
 --------------------------------
 --------------------------------
 READ-FILE-AS-STRING-NON-EXISTING-FILE in READ-FILE-AS-STRING []: 
      Unexpected Error: #<UNDEFINED-FUNCTION READ-FILE-AS-STRING {11057B96A3}>
The function MY-FIVEAM-TEST::READ-FILE-AS-STRING is undefined..
 --------------------------------
 --------------------------------
 READ-FILE-AS-STRING-EMPTY-FILE in READ-FILE-AS-STRING []: 
      Unexpected Error: #<UNDEFINED-FUNCTION READ-FILE-AS-STRING {11057B8073}>
The function MY-FIVEAM-TEST::READ-FI

# Fixtures

In [3]:
(in-package :cl-user)
(defpackage my-fiveam-test-fixtures
  (:use :cl
        :fiveam))
(in-package :my-fiveam-test-fixtures)

(def-suite my-system-fixtures
  :description "Test my system")
(in-suite my-system-fixtures)

(gen-float)
(funcall (gen-float))
(funcall (gen-integer :max 27 :min -16))


(test randomtest
  (for-all ((a (gen-integer :min 1 :max 10))
            (b (gen-integer :min 1 :max 10)))
    "Test random tests."
    (is (<= a b))))

(run! 'randomtest)


#<PACKAGE "COMMON-LISP-USER">

#<PACKAGE "MY-FIVEAM-TEST-FIXTURES">

#<PACKAGE "MY-FIVEAM-TEST-FIXTURES">

MY-SYSTEM-FIXTURES

MY-SYSTEM-FIXTURES

...

RANDOMTEST

NIL

(#<IT.BESE.FIVEAM::FOR-ALL-TEST-FAILED {1101642E93}>)

NIL


Running test RANDOMTEST .ff
 Did 1 check.
    Pass: 0 ( 0%)
    Skip: 0 ( 0%)
    Fail: 1 (100%)

 Failure Details:
 --------------------------------
 RANDOMTEST in MY-SYSTEM-FIXTURES []: 
      Falsifiable with (10 3)
 Results collected with failure data:
    Did 1 check.
       Pass: 0 ( 0%)
       Skip: 0 ( 0%)
       Fail: 1 (100%)

    Failure Details:
    --------------------------------
    RANDOMTEST in MY-SYSTEM-FIXTURES []: 
         
B

 evaluated to 

3

 which is not 

<=

 to 

10


    --------------------------------

 --------------------------------



# Cleanup

!rm 

# End